# Non-Euclidean Geometries: Hyperbolic Poincaré, Spherical, and SPD Manifolds

[![Track](https://img.shields.io/badge/Track-PyTorch_2.x_%7C_JAX_%7C_Candle_%7C_Kernels-blue.svg)](../../README.md)
[![Parity](https://img.shields.io/badge/Parity-Transitive_%3C_1e--5-success.svg)](../../tests/test_cross_track_parity.py)

### Historical Lineage & Seminal Citations
- Lobachevsky (1829) & Poincaré (1882): Discovered hyperbolic non-Euclidean geometry with constant negative curvature.
- Nickel & Kiela (NeurIPS 2017, Poincaré Embeddings): Showed that 2D Poincaré space embeds exponential tree hierarchies with orders of magnitude lower distortion than Euclidean spaces.
- Wang & Isola (ICML 2020, Alignment and Uniformity on the Hypersphere): Formulated contrastive representation learning as optimizing alignment and uniformity on unit sphere S^(d-1).
- Arsigny et al. (MRM 2006, Log-Euclidean Metrics on SPD Tensors): Developed the Log-Euclidean Lie group metric for fast affine-invariant computations on symmetric positive definite manifolds.

### Problem Statement & Purpose
Flat Euclidean space (R^d) expands polynomially (V(r) ~ r^d), causing severe distortion and dimensional crowding when embedding hierarchical taxonomies or directional data. This notebook demonstrates how hyperbolic Poincaré balls embed exponential trees, how unit hyperspheres isolate directional semantic contrast, and how SPD manifolds process covariance representations via the Log-Euclidean metric.

### Decision Boundary: When NOT to Use
> [!WARNING]
> Do NOT optimize Poincaré ball embeddings with unrescaled Euclidean gradients; Euclidean steps near the boundary correspond to vast geodesic distances and push points outside the disk (||x|| >= 1/sqrt(c)), causing artanh(>=1) = NaN! Always scale gradients by the conformal factor (1 - c||x||^2)^2 / 4 and clamp radius. Do NOT use Euclidean distances to measure tree hierarchies or graph taxonomies; volume crowding forces unnecessary dimensional explosion.

### Topic Connections
🔗 Upstream: [08. Computation Spaces & JEPA](08_computation_spaces_latent_reasoning_and_jepa.ipynb) | 🔗 Downstream: [10. Decision Hierarchy L0-L5](10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.ipynb)


## Mathematical Foundations & Riemannian Operations

### 1. Poincaré Ball Model (\mathbb{D}_c^d)
The Poincaré ball of curvature $c > 0$ has conformal factor $\lambda_x^c = \frac{2}{1 - c\|\mathbf{x}\|^2}$.
$$\mathbf{x} \oplus_c \mathbf{y} = \frac{(1 + 2c\langle\mathbf{x},\mathbf{y}\rangle + c\|\mathbf{y}\|^2)\mathbf{x} + (1 - c\|\mathbf{x}\|^2)\mathbf{y}}{1 + 2c\langle\mathbf{x},\mathbf{y}\rangle + c^2\|\mathbf{x}\|^2\|\mathbf{y}\|^2}$$
$$d_{\mathbb{D}}^c(\mathbf{x}, \mathbf{y}) = \frac{2}{\sqrt{c}} \text{artanh}\left(\sqrt{c} \|(-\mathbf{x}) \oplus_c \mathbf{y}\|\right)$$

### 2. Hyperspherical Geometry (\mathbb{S}^{d-1}) & SPD Manifold (\mathcal{S}_{++}^d)
$$d_{\mathbb{S}}(\mathbf{p}, \mathbf{q}) = \arccos(\langle\mathbf{p}, \mathbf{q}\rangle), \quad d_{\text{LE}}(\mathbf{P}, \mathbf{Q}) = \|\log(\mathbf{P}) - \log(\mathbf{Q})\|_F$$

### Asymptotic Complexity
- Poincaré & Spherical Distance: $\mathcal{O}(d)$ compute per pair.
- SPD Log-Euclidean ($d=2$): $\mathcal{O}(1)$ closed-form spectral evaluation.


In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

repo_root = Path.cwd()
while not (repo_root / "core").is_dir() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core  # noqa: E402

core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_path = repo_root / "weights" / "01_paradigms_and_dichotomies" / "09_non_euclidean_geometries_hyperbolic_poincare_spherical_and_spd.safetensors"
weights, ref_in, ref_out, meta = core.parity.load_reference_checkpoint(weights_path, device=device)
print(f"Device: {device} | Checkpoint: {weights_path.name} | Meta: {meta}")


In [ ]:
from jaxtyping import Float
from torch import Tensor

class PoincareBall:
    @staticmethod
    def exp_map_zero(v: Float[Tensor, "b d"], c: float = 1.0) -> Float[Tensor, "b d"]:
        sqrt_c = c ** 0.5
        v_norm = torch.norm(v, p=2, dim=-1, keepdim=True).clamp_min(1e-8)
        return (torch.tanh(sqrt_c * v_norm / 2.0) / (sqrt_c * v_norm)) * v

    @staticmethod
    def log_map_zero(x: Float[Tensor, "b d"], c: float = 1.0) -> Float[Tensor, "b d"]:
        sqrt_c = c ** 0.5
        x_norm = torch.norm(x, p=2, dim=-1, keepdim=True).clamp_min(1e-8)
        return (2.0 * torch.atanh(torch.clamp(sqrt_c * x_norm, max=0.9999)) / (sqrt_c * x_norm)) * x

    @staticmethod
    def mobius_add(x: Float[Tensor, "b d"], y: Float[Tensor, "b d"], c: float = 1.0) -> Float[Tensor, "b d"]:
        xy_inner = torch.sum(x * y, dim=-1, keepdim=True)
        x_sq, y_sq = torch.sum(x ** 2, dim=-1, keepdim=True), torch.sum(y ** 2, dim=-1, keepdim=True)
        num = (1.0 + 2.0 * c * xy_inner + c * y_sq) * x + (1.0 - c * x_sq) * y
        denom = 1.0 + 2.0 * c * xy_inner + (c ** 2) * x_sq * y_sq
        return num / denom.clamp_min(1e-15)

    @classmethod
    def distance(cls, x: Float[Tensor, "b d"], y: Float[Tensor, "b d"], c: float = 1.0) -> Float[Tensor, "b"]:
        diff = cls.mobius_add(-x, y, c=c)
        diff_norm = torch.norm(diff, p=2, dim=-1).clamp(max=0.9999)
        return (2.0 / (c ** 0.5)) * torch.atanh(diff_norm)

class SphericalManifold:
    @staticmethod
    def distance(p: Float[Tensor, "b d"], q: Float[Tensor, "b d"]) -> Float[Tensor, "b"]:
        return torch.acos(torch.sum(p * q, dim=-1).clamp(-1.0 + 1e-7, 1.0 - 1e-7))

    @staticmethod
    def alignment(p: Float[Tensor, "b d"], q: Float[Tensor, "b d"]) -> Float[Tensor, "b"]:
        return torch.sum((p - q) ** 2, dim=-1)

class SPDManifold:
    @staticmethod
    def log_euclidean_distance(P: Float[Tensor, "b d d"], Q: Float[Tensor, "b d d"]) -> Float[Tensor, "b"]:
        def mat_log(M):
            L, Q_m = torch.linalg.eigh(M)
            log_L = torch.diag_embed(torch.log(L.clamp_min(1e-6)))
            return torch.matmul(Q_m, torch.matmul(log_L, Q_m.transpose(-1, -2)))
        return torch.norm(mat_log(P) - mat_log(Q), p="fro", dim=(-2, -1))

print("PoincareBall, SphericalManifold, and SPDManifold operations initialized.")


In [ ]:
# 1. Canonical Linear Anchor Parity
w_anchor, b_anchor = weights["weight"], weights["bias"]
y_linear = ref_in @ w_anchor + b_anchor
assert float(torch.max(torch.abs(y_linear - ref_out)).item()) < 1e-5

# 2. Poincaré Ball Parity & Invariants
c_val = float(weights["c"].item())
v, x_hyp, y_hyp = weights["euclidean_v"], weights["poincare_x"], weights["poincare_y"]
exp_calc = PoincareBall.exp_map_zero(v, c=c_val)
log_calc = PoincareBall.log_map_zero(x_hyp, c=c_val)
mobius_calc = PoincareBall.mobius_add(x_hyp, y_hyp, c=c_val)
dist_calc = PoincareBall.distance(x_hyp, y_hyp, c=c_val)

assert float(torch.max(torch.abs(exp_calc - weights["exp_map_0"])).item()) < 1e-5
assert float(torch.max(torch.abs(log_calc - weights["log_map_0"])).item()) < 1e-5
assert float(torch.max(torch.abs(mobius_calc - weights["mobius_add"])).item()) < 1e-5
assert float(torch.max(torch.abs(dist_calc - weights["poincare_dist"])).item()) < 1e-5
# Invariant: Self-distance is zero
assert float(torch.max(PoincareBall.distance(x_hyp, x_hyp, c=c_val)).item()) < 1e-5

# 3. Spherical Geometry Parity & Invariants
p, q = weights["sphere_p"], weights["sphere_q"]
s_dist_calc = SphericalManifold.distance(p, q)
align_calc = SphericalManifold.alignment(p, q)
assert float(torch.max(torch.abs(s_dist_calc - weights["sphere_dist"])).item()) < 1e-5
assert float(torch.max(torch.abs(align_calc - weights["alignment_loss"])).item()) < 1e-5

# 4. SPD Log-Euclidean Distance Parity
P_spd, Q_spd = weights["spd_P"], weights["spd_Q"]
spd_dist_calc = SPDManifold.log_euclidean_distance(P_spd, Q_spd)
assert float(torch.max(torch.abs(spd_dist_calc - weights["log_euclidean_dist"])).item()) < 1e-5
print("All mathematical invariants (Poincaré metric, Spherical distance, SPD Log-Euclidean) verified.")


In [ ]:
# Multi-Track Micro-Benchmark Table
track_fns = {
    "Linear Anchor Projection": lambda: ref_in @ w_anchor + b_anchor,
    "Poincaré Exp Map": lambda: PoincareBall.exp_map_zero(v, c=c_val),
    "Poincaré Möbius Add": lambda: PoincareBall.mobius_add(x_hyp, y_hyp, c=c_val),
    "Poincaré Distance": lambda: PoincareBall.distance(x_hyp, y_hyp, c=c_val),
    "Spherical Distance": lambda: SphericalManifold.distance(p, q),
    "SPD Log-Euclidean Dist": lambda: SPDManifold.log_euclidean_distance(P_spd, Q_spd),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Misuse: Unprojected Boundary Violation vs Safe Metric Clamping
# Demonstrates boundary singularity when points step outside ||x|| >= 1/sqrt(c)
x_outside = torch.tensor([[1.05, 0.0]], device=device)
nan_produced = torch.isnan(torch.atanh(x_outside[:, 0])).item()
print(f"[FAILURE DEMONSTRATED] Unprojected step ||x||=1.05 produced NaN: {nan_produced}!")
assert nan_produced, "Boundary violation failed to produce NaN"

# Safe Pattern: Boundary Projection
def safe_project(x: Tensor, c: float = 1.0, eps: float = 1e-5) -> Tensor:
    max_norm = (1.0 - eps) / (c ** 0.5)
    norm = torch.norm(x, p=2, dim=-1, keepdim=True).clamp_min(1e-15)
    return torch.where(norm > max_norm, x / norm * max_norm, x)

x_safe = safe_project(x_outside)
assert not torch.isnan(torch.atanh(x_safe[:, 0])).item(), "Safe projected point still NaN"
print(f"[CORRECT] Boundary projection clamped norm to {torch.norm(x_safe).item():.4f} preventing NaN.")

class LinearWrapper(nn.Module):
    def __init__(self, w, b): super().__init__(); self.w, self.b = nn.Parameter(w), nn.Parameter(b)
    def forward(self, x): return x @ self.w + self.b
model_wrap = LinearWrapper(w_anchor, b_anchor).to(device)
ok_exp, _ = core.export.verify_torch_export(model_wrap, ref_in)
ok_comp, _ = core.export.verify_torch_compile_fullgraph(model_wrap, ref_in)
assert ok_exp and ok_comp, "Export or compilation failed"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.")


### Mathematical & Architectural Synthesis

<svg viewBox="0 0 650 120" width="100%" height="120" xmlns="http://www.w3.org/2000/svg">
  <rect x="5" y="5" width="640" height="110" rx="8" fill="#0f172a" stroke="#334155" stroke-width="1.5"/>
  <rect x="25" y="25" width="135" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="92" y="47" fill="#38bdf8" font-size="11" font-weight="bold" text-anchor="middle">Poincaré Ball D_c</text>
  <rect x="175" y="25" width="135" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="242" y="47" fill="#10b981" font-size="11" font-weight="bold" text-anchor="middle">Spherical S^(d-1)</text>
  <rect x="325" y="25" width="135" height="35" rx="6" fill="#1e293b" stroke="#f59e0b" stroke-width="1.5"/>
  <text x="392" y="47" fill="#f59e0b" font-size="11" font-weight="bold" text-anchor="middle">SPD S++^d</text>
  <rect x="475" y="25" width="150" height="35" rx="6" fill="#1e293b" stroke="#a855f7" stroke-width="1.5"/>
  <text x="550" y="47" fill="#a855f7" font-size="11" font-weight="bold" text-anchor="middle">Riemannian SGD</text>
  <rect x="25" y="70" width="600" height="32" rx="4" fill="#0f172a" stroke="#64748b" stroke-width="1.2" stroke-dasharray="4 4"/>
  <text x="325" y="90" fill="#cbd5e1" font-size="10" text-anchor="middle">Hyperbolic: Tree hierarchies | Spherical: Contrastive angles | SPD: Covariances | Rescaled grads</text>
</svg>

### Key Algorithmic Takeaways
1. **Volume Matching:** Exponential tree branching matches hyperbolic volume growth V(r) ~ e^(sqrt(c)r), eliminating Euclidean dimensional crowding.
2. **Conformal Metric Rescaling:** Riemannian gradient updates require scaling by the inverse conformal factor (1 - c||x||^2)^2 / 4.
3. **Boundary Safety:** Projecting points strictly within ||x|| <= (1 - eps)/sqrt(c) mathematically prevents artanh domain errors.
4. **Log-Euclidean Lie Algebra:** Matrix logarithms linearize SPD manifolds into vector space operations.

---
**Curriculum Navigation:**
&larr; [Previous: 08. Computation Spaces & JEPA](08_computation_spaces_latent_reasoning_and_jepa.ipynb) | [Master Index: README.md](../../README.md) | Next: 10. Decision Hierarchy L0-L5 (Upcoming in Category 01) &rarr;
